# AEDH-LSTM Training Notebook

基于 `src/alpha_arena/train/main.py` 和 `src/notebooks/train_aedh_lstm.py` 的单机训练 notebook。运行下面的代码单元即可加载数据、开始训练、查看日志，并绘制 loss 曲线。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find project root from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOK_DIR = SRC_DIR / "notebooks"

for path in (SRC_DIR, NOTEBOOK_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from alpha_arena.models.aedh_lstm import AttentionEnhancedDualHeadLSTM
from alpha_arena.train.dataset.loader import SequenceDataset
from alpha_arena.train.trainer import train_model_ddp
from alpha_arena.utils import configure_logging, get_logger
from train_aedh_lstm import (
    DEFAULT_CHECKPOINT_DIR,
    DEFAULT_DATASET_NAME,
    create_notebook_dataloaders,
    infer_model_config,
    plot_loss_curves,
    resolve_device,
)

configure_logging(log_dir=PROJECT_ROOT / "logs", file_name="train_notebook.log")
logger = get_logger("notebooks.train_aedh_lstm")
pd.set_option("display.max_columns", None)
PROJECT_ROOT

In [ ]:
dataset_name = DEFAULT_DATASET_NAME
batch_size = 512
num_workers = 4
num_epochs = 50
warmup_epochs = 10
mid_epochs = 0
lr = 1e-3
weight_decay = 0.0
patience = 10
max_grad_norm = None
device = resolve_device("auto")
checkpoint_dir = PROJECT_ROOT / DEFAULT_CHECKPOINT_DIR
plot_path = checkpoint_dir / "loss_curve.png"

pin_memory = device.type == "cuda"

logger.info(
    "notebook_configured",
    dataset_name=dataset_name,
    batch_size=batch_size,
    num_workers=num_workers,
    num_epochs=num_epochs,
    device=str(device),
    checkpoint_dir=str(checkpoint_dir),
)

{
    "dataset_name": dataset_name,
    "device": str(device),
    "checkpoint_dir": str(checkpoint_dir),
}

In [ ]:
logger.info("loading_dataset", dataset_name=dataset_name)
train_dataset = SequenceDataset(dataset_name=dataset_name, split_name="train")
valid_dataset = SequenceDataset(dataset_name=dataset_name, split_name="evaluate")

model_config = infer_model_config(train_dataset)
logger.info(
    "dataset_loaded",
    train_size=len(train_dataset),
    valid_size=len(valid_dataset),
    input_dim=model_config.input_dim,
    cs_feature_dim=model_config.cs_feature_dim,
)

dataloaders = create_notebook_dataloaders(
    train_dataset=train_dataset,
    valid_dataset=valid_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

model = AttentionEnhancedDualHeadLSTM(model_config)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=lr,
    weight_decay=weight_decay,
)

logger.info("start_training", checkpoint_dir=str(checkpoint_dir), plot_path=str(plot_path))
result = train_model_ddp(
    model=model,
    optimizer=optimizer,
    train_random_loader=dataloaders["train_random_loader"],
    train_grouped_loader=dataloaders["train_grouped_loader"],
    valid_loader=dataloaders["valid_loader"],
    device=device,
    num_epochs=num_epochs,
    warmup_epochs=warmup_epochs,
    mid_epochs=mid_epochs,
    patience=patience,
    max_grad_norm=max_grad_norm,
    checkpoint_dir=checkpoint_dir,
    use_ddp=False,
    verbose=True,
)

history = result["history"] or []
history_frame = plot_loss_curves(history=history, output_path=plot_path, show=True)
logger.info(
    "training_completed",
    best_epoch=result["best_epoch"],
    best_metric=result["best_metric"],
    history_path=str(checkpoint_dir / "history.json"),
    plot_path=str(plot_path),
)

display(history_frame.tail())

In [ ]:
summary_columns = ["epoch", "stage", "train/loss", "valid/loss", "lr"]
display(history_frame[summary_columns].tail(10))
print(f"best_epoch={result['best_epoch']}, best_metric={result['best_metric']:.6f}")
print(f"loss curve saved to: {plot_path}")